# An Expert-Reviewed NDVI-Soil Index Framework Using Multispectral UAV Imagery

This notebook implements the reproducible analysis pipeline accompanying the manuscript: NDVI and Soil Index pseudo-labels, U-Net probability segmentation, and XGBoost tile classification.

The reported metrics measure agreement with rule-derived pseudo-labels; they are not independent field-ground-truth accuracy. See the repository README and manuscript for interpretation limits.


# Section 1 — Setup Environment & Konfigurasi Path

**Deskripsi:** Mount Google Drive (Colab), install dependensi, deteksi runtime Colab vs lokal, dan set path dataset/output. Semua path output dibuat otomatis.

---


In [ ]:
# Install dependencies when running in Colab or a fresh environment.
%pip install -q tifffile tqdm albumentations segmentation-models-pytorch torch torchvision xgboost scikit-learn scipy opencv-python-headless scikit-image seaborn pandas pillow


In [ ]:
# === Google Drive Mount (Colab) ===
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted successfully.')
except ModuleNotFoundError:
    print('Running outside Google Colab (Drive not mounted).')

In [ ]:
# All Import Package and Konstanta
# ── Standard library ─────────────────────────────────────────────────
import seaborn as sns
from skimage.feature import graycomatrix, graycoprops
from sklearn.metrics import (
    accuracy_score,
    accuracy_score as sk_acc,
    precision_recall_fscore_support,

    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
import xgboost
import albumentations as A
import segmentation_models_pytorch as smp
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch
import matplotlib.pyplot as plt
%matplotlib inline
import os
import sys
import gc
import ctypes
import random as _random
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

# ── Third-party: core ───────────────────────────────────────────────
import numpy as np
import pandas as pd
import cv2
import tifffile as tiff
from PIL import Image as PILImage

# ── Matplotlib (Agg for batch save; display via IPython) ─────────────
import matplotlib
matplotlib.use('Agg')

# ── Progress bar (Colab notebook / fallback std tqdm) ─────────────────
try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

# ── ML / DL ───────────────────────────────────────────────────────────

# ── IPython display (never shadow PIL.Image) ──────────────────────────
try:
    from IPython.display import Image as IPyImage, display
except ImportError:
    IPyImage, display = None, print

# ── Global constants ──────────────────────────────────────────────────
EPS = 1e-10
BIT_DEPTH = 65535

CLASS_NAMES = ['Sehat', 'Rusak Ringan', 'Rusak Sedang', 'Rusak Parah']
CLASS_COLORS = np.array([
    [0, 180, 0],
    [255, 220, 0],
    [255, 140, 0],
    [220, 0, 0],
], dtype=np.uint8)

THRESHOLDS = {
    'ndvi_healthy': 0.50,
    'ndvi_light': 0.35,
    'ndvi_moderate': 0.20,
    'si_light': 0.10,
    'si_moderate': 0.20,
    'si_severe': 0.35,
}


PATCH_SIZE = 256
STRIDE = 128
IN_CHANNELS = 2
NUM_CLASSES = 4

# === V2 cascade feature constants ===
PROB_CLASS_KEYS = ['sehat', 'ringan', 'sedang', 'parah']  # selaras CLASS_NAMES
N_BASE_FEATURES = 14
N_UNET_PROB_FEATURES = 10
N_CASCADE_FEATURES = N_BASE_FEATURES + N_UNET_PROB_FEATURES  # 24
RANDOM_SEED = 42
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15
BATCH_SIZE = 8 if torch.cuda.is_available() else 2
EPOCHS = 10
LR = 1e-4

VEG_KEYS = ['NDVI', 'NDRE', 'GNDVI']
SOIL_KEYS = ['SI', 'BSI', 'CI', 'BI']
SAVE_PNG = True
FLUSH_EVERY = 20

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPS_T = torch.tensor(EPS, dtype=torch.float32, device=DEVICE)

print('All packages imported.')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# import shutil
# import os

# # Change this path to the folder you want to delete
# folder_to_delete = '/content/WORKSPACE_DATA/ANALYSIS_RESULTS'

# if os.path.exists(folder_to_delete):
#     shutil.rmtree(folder_to_delete)
#     print(f"Successfully deleted: {folder_to_delete}")
# else:
#     print(f"Folder not found: {folder_to_delete}")


In [ ]:
IN_COLAB = 'google.colab' in sys.modules

# Set LAND_DEGRADATION_DATA_DIR to override the default data workspace.
if IN_COLAB:
    default_base_dir = Path('/content/WORKSPACE_DATA')
else:
    default_base_dir = Path.cwd() / 'data'

BASE_DIR = Path(os.environ.get('LAND_DEGRADATION_DATA_DIR', default_base_dir)).expanduser()

src_folder = str(BASE_DIR / 'DJI_202605021610_001_Create-Area-Route2')
dst_folder = str(BASE_DIR / 'OUTPUT')
analysis_dir = str(BASE_DIR / 'ANALYSIS_RESULTS')
masks_dir = str(BASE_DIR / 'masks')
dataset_dir = str(BASE_DIR / 'dataset')
models_dir = str(BASE_DIR / 'models')
reports_dir = str(BASE_DIR / 'reports')

for d in [analysis_dir, masks_dir, dataset_dir, models_dir, reports_dir]:
    os.makedirs(d, exist_ok=True)

def get_dir_size_gb(dir_path):
    total_size = 0
    if os.path.exists(dir_path):
        for dirpath, _, filenames in os.walk(dir_path):
            for filename in filenames:
                file_path = os.path.join(dirpath, filename)
                if not os.path.islink(file_path):
                    total_size += os.path.getsize(file_path)
    return total_size / (1024**3)

print(f'Runtime: {"Colab" if IN_COLAB else "Local"}')
print(f'BASE_DIR: {BASE_DIR}')
print('Set LAND_DEGRADATION_DATA_DIR if your workspace is elsewhere.')


# Section 2 — Eksplorasi & Validasi Dataset

**Deskripsi:** Inventaris file sumber dan folder scene multispectral. Verifikasi band NIR tersedia (437 scene × 5 file: JPG + 4 band TIF). Output: statistik file dan contoh struktur folder.

---


In [ ]:
if os.path.exists(src_folder):
    files = os.listdir(src_folder)
    file_types = Counter(
        os.path.splitext(f)[1].lower()
        for f in files
        if os.path.isfile(os.path.join(src_folder, f))
    )
    print(f'Source folder: {len(files)} files')
    for ext, count in file_types.items():
        print(f'  - {ext or "no ext"}: {count}')
else:
    print(f'Source folder not found: {src_folder}')

In [ ]:
if not os.path.exists(dst_folder):
    print(f'Folder not found: {dst_folder}')
    scene_folders = []
else:
    subfolders = sorted(
        f for f in os.listdir(dst_folder)
        if os.path.isdir(os.path.join(dst_folder, f))
    )
    all_files = []
    for root, _, files in os.walk(dst_folder):
        all_files.extend(files)
    file_patterns = Counter(os.path.splitext(f)[1].lower() for f in all_files)

    print(f'--- Analysis for {dst_folder} ---')
    print(f'Total Scene Folders: {len(subfolders)}')
    print(f'Total Files Found: {len(all_files)}')
    print('File Extension Patterns:')
    for ext, count in file_patterns.items():
        print(f'  - {ext or "no ext"}: {count}')

    print('\nSample Folder Structure (first 3 scenes):')
    for folder in subfolders[:3]:
        content = sorted(os.listdir(os.path.join(dst_folder, folder)))
        print(f'  {folder}/')
        for item in content[:5]:
            print(f'    - {item}')

    scene_folders = sorted(d for d in subfolders if d.startswith('Scene_'))
    print(f'\nScene folders: {len(scene_folders)}')
    HAS_NIR = True
    print(f'NIR sensor available: {HAS_NIR} (real NDVI enabled)')

## 2.1 — Statistik Dimensi & Ukuran File

**Deskripsi:** Cek dimensi (height × width) tiap band TIF dan ukuran file untuk memastikan konsistensi spasial antar scene. Bila ada scene dengan dimensi berbeda, perlu resampling sebelum stacking.


In [ ]:
_dim_rows = []
_sample_scenes = scene_folders[: min(20, len(scene_folders))]
for _scene in _sample_scenes:
    _spath = os.path.join(dst_folder, _scene)
    _row = {'Scene': _scene}
    for _f in sorted(os.listdir(_spath)):
        _full = os.path.join(_spath, _f)
        _ext = os.path.splitext(_f)[1].lower()
        try:
            if _ext == '.tif':
                _arr = tiff.imread(_full)
                _key = next(
                    (b for b in ['MS_NIR', 'MS_RE', 'MS_G', 'MS_R'] if b in _f.upper()), 'TIF')
                _row[f'{_key}_shape'] = f'{_arr.shape[0]}x{_arr.shape[1]}'
                _row[f'{_key}_dtype'] = str(_arr.dtype)
            elif _ext == '.jpg':
                with PILImage.open(_full) as _im:
                    _row['RGB_shape'] = f'{_im.size[1]}x{_im.size[0]}'
            _row[f'{_f}_size_kb'] = round(os.path.getsize(_full) / 1024, 1)
        except Exception as _e:
            _row[_f] = f'ERR: {_e}'
    _dim_rows.append(_row)

df_dim = pd.DataFrame(_dim_rows)
_shape_cols = [c for c in df_dim.columns if c.endswith('_shape')]
print('Sample dimensions (first 20 scenes):')
print(df_dim[['Scene'] + _shape_cols].to_string(index=False))
print('\nUnique shapes per band:')
for _c in _shape_cols:
    print(f'  {_c}: {df_dim[_c].dropna().unique().tolist()}')

## 2.2 — Statistik Intensitas Band (Mean / Std / Min / Max)

**Deskripsi:** Hitung statistik intensitas pixel per band (G, R, RE, NIR) pada sample scene untuk memahami distribusi raw 16-bit. Berguna untuk menentukan normalisasi yang tepat dan deteksi band yang corrupt / clipped.


In [ ]:
def load_multispectral(scene_folder_path):
    bands = {}
    files = os.listdir(scene_folder_path)
    tif_files = [f for f in files if f.lower().endswith('.tif')]
    suffix_map = {
        'MS_NIR': 'NIR',
        'MS_RE': 'RE',
        'MS_G': 'G',
        'MS_R': 'R',
    }
    for f in tif_files:
        f_upper = f.upper()
        for suffix, band_name in suffix_map.items():
            if suffix in f_upper and band_name not in bands:
                path = os.path.join(scene_folder_path, f)
                # FIX: return uint16 mentah tanpa astype — biarkan GPU yang melakukan konversi
                bands[band_name] = tiff.imread(path)
                break
    if len(bands) != 4:
        found = list(bands.keys())
        raise ValueError(
            f'Scene {os.path.basename(scene_folder_path)} hanya {len(found)} band: {found}'
        )
    return bands


# FIX: Fungsi ini tetap dipertahankan namun penggunaannya bisa diskip jika pipeline utama menggunakan raw array
def normalize_bands(bands, bit_depth=BIT_DEPTH):
    return {k: v / float(bit_depth) for k, v in bands.items()}


def load_rgb_jpg(scene_folder_path):
    for f in os.listdir(scene_folder_path):
        if f.lower().endswith('.jpg'):
            img = np.array(PILImage.open(os.path.join(
                scene_folder_path, f)).convert('RGB'))
            return img.astype(np.float32) / 255.0
    return None


In [ ]:
if not scene_folders:
    print('No scene folders found. Run Section 2 dataset inventory cell first.')
    df_band = pd.DataFrame()
else:
    _band_stats = []
    _sample = scene_folders[: min(30, len(scene_folders))]
    for _scene in tqdm(_sample, desc='Band statistics'):
        try:
            _bands_raw = load_multispectral(os.path.join(dst_folder, _scene))
            for _bname, _arr in _bands_raw.items():
                _band_stats.append({
                    'Scene': _scene,
                    'Band': _bname,
                    'Mean': float(_arr.mean()),
                    'Std': float(_arr.std()),
                    'Min': float(_arr.min()),
                    'Max': float(_arr.max()),
                    'NaN%': float(np.isnan(_arr).mean() * 100),
                })
        except Exception as _e:
            print(f'Skip {_scene}: {_e}')

    df_band = pd.DataFrame(_band_stats)
    if not df_band.empty:
        print('Aggregate statistics per band (across sample scenes):')
        print(df_band.groupby('Band')[['Mean', 'Std', 'Min', 'Max']].agg(
            ['mean', 'std']).round(2))
    else:
        print('No data collected. Check if source folders contain valid .tif files.')

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

hist_path = os.path.join(reports_dir, 'band_mean_distribution.png')

if 'df_band' not in dir() or df_band.empty:
    print('Run the band statistics cell above first.')
elif os.path.exists(hist_path):
    print(f'Loading cached visualization from: {hist_path}')
    display(IPyImage(filename=hist_path))
else:
    # Ensure interactive mode for display
    plt.ion()
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    for ax, bname in zip(axes, ['G', 'R', 'RE', 'NIR']):
        sub = df_band[df_band['Band'] == bname]
        ax.hist(sub['Mean'], bins=20, color='steelblue',
                edgecolor='black', alpha=0.8)
        ax.set_title(f'Distribusi Mean Band {bname}')
        ax.set_xlabel('Mean intensity (raw 16-bit)')
        ax.set_ylabel('Jumlah scene')
        ax.grid(True, alpha=0.3)

    plt.suptitle('Distribusi rata-rata intensitas tiap band')
    plt.tight_layout()

    # Save for efficiency
    os.makedirs(reports_dir, exist_ok=True)
    plt.savefig(hist_path, dpi=120, bbox_inches='tight')

    # Explicitly show the plot in the notebook
    plt.show()
    print(f'Visualization saved to: {hist_path}')

## 2.3 — Preview Visual Sample Scene (Multi-Band)

**Deskripsi:** Tampilkan grid preview RGB + 4 band multispectral untuk 4 scene acak. Ini memvalidasi data secara visual dan membantu deteksi anomali (scene blur, awan, cropping aneh).


In [ ]:
import matplotlib.pyplot as plt
import gc
import ctypes
import torch
%matplotlib inline

def _free_ram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    try:
        ctypes.cdll.LoadLibrary('libc.so.6').malloc_trim(0)
    except:
        pass

preview_path = os.path.join(reports_dir, 'multiband_preview_sample.png')

if os.path.exists(preview_path):
    print(f'Loading cached preview from: {preview_path}')
    display(IPyImage(filename=preview_path))
else:
    _random.seed(42)
    _preview_n = min(4, len(scene_folders))
    _preview = _random.sample(scene_folders, _preview_n) if len(
        scene_folders) >= _preview_n else scene_folders

    # Ensure interactive mode for notebook display
    plt.ion()
    fig, axes = plt.subplots(_preview_n, 5, figsize=(20, 4 * _preview_n))
    if _preview_n == 1:
        axes = np.expand_dims(axes, 0)

    for _row, _scene in enumerate(_preview):
        _spath = os.path.join(dst_folder, _scene)
        try:
            _bands_n = normalize_bands(load_multispectral(_spath))
            _rgb = load_rgb_jpg(_spath)
        except Exception as _e:
            print(f'Skip {_scene}: {_e}')
            continue

        if _rgb is not None:
            axes[_row, 0].imshow(np.clip(_rgb, 0, 1))
        axes[_row, 0].set_title(f'{_scene} - RGB')
        axes[_row, 0].axis('off')

        for _i, _bn in enumerate(['G', 'R', 'RE', 'NIR'], start=1):
            axes[_row, _i].imshow(_bands_n[_bn], cmap='gray', vmin=0, vmax=0.5)
            axes[_row, _i].set_title(f'{_bn}')
            axes[_row, _i].axis('off')

        # RAM Efficiency: Clear temporary band data immediately after plotting
        del _bands_n, _rgb
        _free_ram()

    plt.suptitle('Preview multi-band random sample scenes', fontsize=14)
    plt.tight_layout()

    # Save for efficiency
    os.makedirs(reports_dir, exist_ok=True)
    plt.savefig(preview_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Preview saved to: {preview_path}')
    _free_ram()

## 2.4 — Validasi Integritas File

**Deskripsi:** Periksa setiap scene apakah memiliki 4 band TIF lengkap. Scene yang tidak lengkap akan dimasukkan ke daftar bermasalah dan dikeluarkan dari pipeline berikutnya.


In [ ]:
_missing = []
_required_bands = {'MS_G', 'MS_R', 'MS_NIR', 'MS_RE'}
for _scene in scene_folders:
    _spath = os.path.join(dst_folder, _scene)
    _present = set()
    for _f in os.listdir(_spath):
        for _b in _required_bands:
            if _b in _f.upper():
                _present.add(_b)
    _miss = _required_bands - _present
    if _miss:
        _missing.append({'Scene': _scene, 'Missing': sorted(_miss)})

print(f'Total scenes: {len(scene_folders)}')
print(f'Scenes dengan band tidak lengkap: {len(_missing)}')
if _missing:
    for _r in _missing[:10]:
        print(f'  - {_r["Scene"]}: missing {_r["Missing"]}')
else:
    print('Semua scene memiliki band lengkap.')

valid_scenes = [s for s in scene_folders if s not in {
    r['Scene'] for r in _missing}]
print(f'Valid scenes untuk pipeline: {len(valid_scenes)}')

## 2.5 — Distribusi NDVI Awal (Pre-screen Vegetasi)

**Deskripsi:** Hitung NDVI cepat pada sample scene dan plot histogramnya. Distribusi ini memberi gambaran rentang kerusakan/vegetasi sebelum threshold pseudo-label diset di Section 5.


In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

_ndvi_samples = []
_sample_for_hist = valid_scenes[: min(30, len(valid_scenes))] if 'valid_scenes' in dir(
) else scene_folders[: min(30, len(scene_folders))]
for _scene in tqdm(_sample_for_hist, desc='Quick NDVI histogram'):
    try:
        _b = normalize_bands(load_multispectral(
            os.path.join(dst_folder, _scene)))
        _ndvi = (_b['NIR'] - _b['R']) / (_b['NIR'] + _b['R'] + EPS)
        _ndvi_samples.append(_ndvi.flatten()[::50])
    except Exception:
        continue

if _ndvi_samples:
    _all_ndvi = np.concatenate(_ndvi_samples)
    plt.figure(figsize=(10, 4))
    plt.hist(_all_ndvi, bins=80, color='seagreen',
             edgecolor='black', alpha=0.8)
    for _t, _lbl, _c in [(0.5, 'Sehat', 'green'), (0.35, 'Ringan', 'gold'), (0.2, 'Sedang', 'orange')]:
        plt.axvline(_t, color=_c, linestyle='--',
                    label=f'{_lbl} threshold ({_t})')
    plt.xlabel('NDVI')
    plt.ylabel('Frekuensi pixel (sample)')
    plt.title('Distribusi NDVI agregat (sample 30 scenes)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print(f'NDVI range: [{_all_ndvi.min():.3f}, {_all_ndvi.max():.3f}]')
    print(f'Median: {np.median(_all_ndvi):.3f}')
    print(
        f'Pixels NDVI<0.2 (rusak parah): {(_all_ndvi < 0.2).mean()*100:.2f}%')
    print(
        f'Pixels NDVI>0.5 (sehat)      : {(_all_ndvi > 0.5).mean()*100:.2f}%')

# Section 3 — Perhitungan NDVI & Vegetation Indices

**Deskripsi:** Load band multispectral (G, R, RE, NIR), normalisasi 16-bit, hitung NDVI/NDRE/GNDVI. Demo visual 1 scene, batch export `.npy` + PNG dengan resume capability untuk 437 scene.

---


In [ ]:
def compute_vegetation_indices(bands):
    R, G, NIR, RE = bands['R'], bands['G'], bands['NIR'], bands['RE']
    ndvi = (NIR - R) / (NIR + R + EPS)
    ndre = (NIR - RE) / (NIR + RE + EPS)
    gndvi = (NIR - G) / (NIR + G + EPS)
    return {'NDVI': ndvi, 'NDRE': ndre, 'GNDVI': gndvi}


def save_index_heatmap(idx_data, title, save_path, cmap='RdYlGn', vmin=-1, vmax=1):
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(idx_data, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    plt.colorbar(im, ax=ax, fraction=0.046)
    plt.savefig(save_path, dpi=100, bbox_inches='tight')
    plt.close(fig)

# ── Shared cache / validation utilities ──────────────────────────────


def npy_path(base_dir, scene, key):
    return os.path.join(base_dir, f'{scene}_{key}.npy')


def is_valid_npy(path, quick=True):
    """Fast validity check: exists, non-empty, readable, finite sample."""
    if not os.path.isfile(path):
        return False
    if os.path.getsize(path) <= 128:
        return False
    try:
        arr = np.load(path, mmap_mode='r')
        if arr.size == 0:
            return False
        sample = arr.flat[0]
        if not np.isfinite(sample):
            return False
        if not quick:
            if np.all(np.isnan(arr)) or np.any(np.isinf(arr)):
                return False
        return True
    except Exception:
        return False


def scan_scene_keys(scene, base_dir, keys):
    """Return (missing_keys, corrupt_keys) for one scene."""
    missing, corrupt = [], []
    for key in keys:
        path = npy_path(base_dir, scene, key)
        if not os.path.isfile(path):
            missing.append(key)
        elif not is_valid_npy(path):
            corrupt.append(key)
    return missing, corrupt


def keys_to_compute(scene, base_dir, keys):
    """Keys that are missing or corrupt and need (re)computation."""
    missing, corrupt = scan_scene_keys(scene, base_dir, keys)
    return missing + corrupt


def load_index_mean(path):
    arr = np.load(path, mmap_mode='r')
    mean_val = float(np.nanmean(arr))
    del arr
    return mean_val


def audit_index_cache(scenes, base_dir, keys, label=''):
    """Audit all scenes; return summary dict."""
    complete, partial, missing_all, corrupt_files = [], [], [], []
    for scene in scenes:
        miss, corrupt = scan_scene_keys(scene, base_dir, keys)
        if corrupt:
            corrupt_files.extend(npy_path(base_dir, scene, k) for k in corrupt)
        if not miss and not corrupt:
            complete.append(scene)
        elif miss and len(miss) == len(keys):
            missing_all.append((scene, miss))
        else:
            partial.append((scene, miss + corrupt))
    print(f'[{label}] complete={len(complete)} partial={len(partial)} missing_all={len(missing_all)} corrupt_files={len(corrupt_files)}')
    return {
        'complete': complete,
        'partial': partial,
        'missing_all': missing_all,
        'corrupt_files': corrupt_files,
    }

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if scene_folders:
    demo_scene = os.path.join(dst_folder, scene_folders[0])
    bands = normalize_bands(load_multispectral(demo_scene))
    indices = compute_vegetation_indices(bands)

    fig, axes = plt.subplots(3, 3, figsize=(15, 12))
    axes = axes.ravel()
    for i, band_name in enumerate(['G', 'NIR', 'R', 'RE']):
        im = axes[i].imshow(bands[band_name], cmap='gray', vmin=0, vmax=0.5)
        axes[i].set_title(f'Band {band_name}')
        plt.colorbar(im, ax=axes[i], fraction=0.046)
    for j, (name, data) in enumerate(indices.items()):
        im = axes[j + 4].imshow(data, cmap='RdYlGn', vmin=-1, vmax=1)
        axes[j + 4].set_title(name)
        plt.colorbar(im, ax=axes[j + 4], fraction=0.046)
    for k in range(len(indices) + 4, 9):
        axes[k].axis('off')
    plt.suptitle(f'Scene Analysis: {scene_folders[0]}', fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()
else:
    print('No scene folders found — skip demo.')

In [ ]:
os.makedirs(analysis_dir, exist_ok=True)
npy_dir = os.path.join(analysis_dir, 'npy')
os.makedirs(npy_dir, exist_ok=True)

def _compute_veg_gpu(bands_raw, keys):
    results = {}
    with torch.no_grad():
        # Move raw data to GPU and normalize on GPU
        # FIX: Menerima uint16 mentah, lalu konversi + normalisasi langsung di GPU
        NIR = (torch.from_numpy(bands_raw['NIR']).to(DEVICE).float() / float(BIT_DEPTH))
        if 'NDVI' in keys:
            R = (torch.from_numpy(bands_raw['R']).to(DEVICE).float() / float(BIT_DEPTH))
            results['NDVI'] = ((NIR - R) / (NIR + R + EPS)).cpu().numpy()
            del R
        if 'NDRE' in keys:
            RE = (torch.from_numpy(bands_raw['RE']).to(DEVICE).float() / float(BIT_DEPTH))
            results['NDRE'] = ((NIR - RE) / (NIR + RE + EPS)).cpu().numpy()
            del RE
        if 'GNDVI' in keys:
            G = (torch.from_numpy(bands_raw['G']).to(DEVICE).float() / float(BIT_DEPTH))
            results['GNDVI'] = ((NIR - G) / (NIR + G + EPS)).cpu().numpy()
            del G
        del NIR

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return results

def _free_ram():
    plt.close('all')
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    try: ctypes.cdll.LoadLibrary('libc.so.6').malloc_trim(0)
    except: pass

# ── Efficient Cache Audit ───────────────────────────────────────────
veg_stats = {}
todo_scenes, todo_keys_map = [], {}

for name in tqdm(scene_folders, desc='Auditing Vegetation Cache'):
    need = keys_to_compute(name, npy_dir, VEG_KEYS)
    if not need:
        veg_stats[name] = {k: load_index_mean(npy_path(npy_dir, name, k)) for k in VEG_KEYS}
    else:
        todo_scenes.append(name)
        todo_keys_map[name] = need

print(f'Vegetation Cache: {len(veg_stats)} complete | {len(todo_scenes)} to process')

# ── Optimized Vegetation Processing ──────────────────────────────────
if todo_scenes:
    for i, name in enumerate(tqdm(todo_scenes, desc='Processing Vegetation')):
        keys_needed = todo_keys_map[name]
        try:
            # Load raw bands (no CPU normalization)
            bands_raw = load_multispectral(os.path.join(dst_folder, name))
            # Compute entirely on GPU
            indices = _compute_veg_gpu(bands_raw, keys_needed)
            del bands_raw

            for k, data in indices.items():
                np.save(npy_path(npy_dir, name, k), data.astype(np.float32))
                if SAVE_PNG:
                    png_path = os.path.join(analysis_dir, f'{name}_{k}.png')
                    if not os.path.isfile(png_path):
                        save_index_heatmap(data, f'{k} - {name}', png_path)
                        # FIX: tambahkan plt.close('all') langsung setelah setiap pemanggilan save
                        plt.close('all')
                del data
            del indices

            veg_stats[name] = {k: load_index_mean(npy_path(npy_dir, name, k)) for k in VEG_KEYS}

        except Exception as e: print(f' Skip {name}: {e}')
        finally:
            # FIX: pindahkan _free_ram() ke blok finally setiap iterasi di loop veg
            _free_ram()

_free_ram()

# Export Report
veg_df = pd.DataFrame([{'Scene': s, **v} for s, v in veg_stats.items()])
veg_df.to_csv(os.path.join(reports_dir, 'veg_summary.csv'), index=False)

mean_ndvi_list = [veg_stats[s]['NDVI'] for s in scene_folders if s in veg_stats]
scene_names = [s for s in scene_folders if s in veg_stats]
print(f'Vegetation processing finished. Summary saved to {reports_dir}')


In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
if scene_names:
    fig_w = max(12, len(scene_names) * 0.15)
    fig, ax = plt.subplots(figsize=(fig_w, 5))
    ax.plot(scene_names, mean_ndvi_list,
            marker='o', linestyle='-', markersize=2)
    ax.set_xticks(range(len(scene_names)))
    ax.set_xticklabels(scene_names, rotation=90, fontsize=6)
    ax.set_ylabel('Rata-rata NDVI')
    ax.set_title(f'Rata-rata NDVI per Scene (n={len(scene_names)})')
    ax.grid(True, alpha=0.3)
    fig.tight_layout()

    os.makedirs(reports_dir, exist_ok=True)
    trend_path = os.path.join(reports_dir, 'ndvi_trend_per_scene.png')
    fig.savefig(trend_path, dpi=120, bbox_inches='tight')
    plt.show(fig)

    # Agg backend (from processing cell) cannot plt.show() — display saved PNG instead
    try:
        display(Image(filename=trend_path))
    except NameError:
        pass

    print(f'Saved: {trend_path}')
else:
    print('No scene data — run vegetation indices cell first.')

# Section 4 — Perhitungan Soil Index

**Deskripsi:** Hitung indeks tanah dari band multispectral: Basic Soil Index (SI), Bare Soil Index (BSI), Color Index (CI), dan Brightness Index (BI) dengan proxy Green karena band Blue tidak tersedia. Output: peta grayscale dan file `.npy` normalisasi.

---


In [ ]:
def compute_soil_indices(bands):
    R, G, NIR = bands['R'], bands['G'], bands['NIR']
    si = (R - G) / (R + G + EPS)
    ci = si.copy()
    bsi = ((R + G) - NIR) / ((R + G) + NIR + EPS)
    # FIX: ganti dengan R**2 + G**2 + NIR**2
    bi = np.sqrt(R ** 2 + G ** 2 + NIR ** 2)
    bi_norm = (bi - bi.min()) / (bi.max() - bi.min() + EPS)
    return {'SI': si, 'BSI': bsi, 'CI': ci, 'BI': bi_norm}


def normalize_index(arr):
    arr = arr.astype(np.float32)
    lo, hi = np.nanpercentile(arr, [2, 98])
    return np.clip((arr - lo) / (hi - lo + EPS), 0, 1)


def _free_ram():
    plt.close('all')
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    try: ctypes.cdll.LoadLibrary('libc.so.6').malloc_trim(0)
    except: pass


In [ ]:
soil_npy_dir = os.path.join(analysis_dir, 'soil_npy')
os.makedirs(soil_npy_dir, exist_ok=True)

def _compute_soil_gpu(bands_raw, keys):
    """Compute soil indices using PyTorch on GPU for RAM efficiency and speed."""
    results = {}
    with torch.no_grad():
        # FIX: Menerima uint16 mentah, lalu konversi + normalisasi langsung di GPU
        R = (torch.from_numpy(bands_raw['R']).to(DEVICE).float() / float(BIT_DEPTH))
        G = (torch.from_numpy(bands_raw['G']).to(DEVICE).float() / float(BIT_DEPTH))
        if 'SI' in keys or 'CI' in keys:
            si = (R - G) / (R + G + EPS)
            if 'SI' in keys: results['SI'] = si.cpu().numpy()
            if 'CI' in keys: results['CI'] = si.cpu().numpy()
            del si
        if 'BSI' in keys or 'BI' in keys:
            # FIX: Prepare NIR properly here if it's needed for BI and BSI
            NIR = (torch.from_numpy(bands_raw['NIR']).to(DEVICE).float() / float(BIT_DEPTH))
        if 'BSI' in keys:
            results['BSI'] = (((R + G) - NIR) / ((R + G) + NIR + EPS)).cpu().numpy()
        if 'BI' in keys:
            # FIX: ganti dengan R**2 + G**2 + NIR**2
            bi = torch.sqrt(R ** 2 + G ** 2 + NIR ** 2)
            results['BI'] = ((bi - bi.min()) / (bi.max() - bi.min() + EPS)).cpu().numpy()
            del bi
        if 'BSI' in keys or 'BI' in keys:
            del NIR
        del R, G

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return results

# ── Optimized Soil Audit & Processing ───────────────────────────────
soil_stats_by_scene = {}
todo_soil, todo_soil_keys = [], {}



for scene_name in tqdm(scene_folders, desc='Scanning Soil Cache'):
    need = keys_to_compute(scene_name, soil_npy_dir, SOIL_KEYS)
    if not need:
        soil_stats_by_scene[scene_name] = {
            'SI_mean': load_index_mean(npy_path(soil_npy_dir, scene_name, 'SI')),
            'BSI_mean': load_index_mean(npy_path(soil_npy_dir, scene_name, 'BSI')),
        }
    else:
        todo_soil.append(scene_name)
        todo_soil_keys[scene_name] = need

print(f'Soil Cache: {len(soil_stats_by_scene)} complete | {len(todo_soil)} to process')

if todo_soil:
    for i, scene_name in enumerate(tqdm(todo_soil, desc='Processing Soil Indices')):
        keys_needed = todo_soil_keys[scene_name]
        try:
            scene_path = os.path.join(dst_folder, scene_name)
            bands_raw = load_multispectral(scene_path)
            # FIX: hapus pemanggilan normalize_bands dan oper uint16 mentah
            soil_indices = _compute_soil_gpu(bands_raw, keys_needed)
            del bands_raw

            for name, data in soil_indices.items():
                out_path = npy_path(soil_npy_dir, scene_name, name)
                np.save(out_path, data.astype(np.float32))

                if SAVE_PNG:
                    png_path = os.path.join(analysis_dir, f'{scene_name}_{name}.png')
                    if not os.path.isfile(png_path):
                        save_index_heatmap(normalize_index(data), f'{name} - {scene_name}', png_path, cmap='gray', vmin=0, vmax=1)
                        # FIX: tambahkan plt.close('all') langsung setelah setiap pemanggilan save
                        plt.close('all')
                del data
            del soil_indices

            # Update stats
            soil_stats_by_scene[scene_name] = {
                'SI_mean': load_index_mean(npy_path(soil_npy_dir, scene_name, 'SI')),
                'BSI_mean': load_index_mean(npy_path(soil_npy_dir, scene_name, 'BSI')),
            }

        except Exception as e:
            print(f'  Skip {scene_name}: {e}')
        finally:
            # FIX: pindahkan _free_ram() ke blok finally setiap iterasi di loop soil
            _free_ram()

_free_ram()

# Save summary CSV
soil_summary = pd.DataFrame([{'Scene': s, **v} for s, v in soil_stats_by_scene.items()])
soil_csv_path = os.path.join(reports_dir, 'soil_indices_summary.csv')
soil_summary.to_csv(soil_csv_path, index=False)
print(f'\nSoil processing finished. Summary saved to {soil_csv_path}')


In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
if scene_folders:
    demo = os.path.join(dst_folder, scene_folders[0])
    bands = normalize_bands(load_multispectral(demo))
    soil = compute_soil_indices(bands)

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for ax, (name, data) in zip(axes, soil.items()):
        im = ax.imshow(normalize_index(data), cmap='gray', vmin=0, vmax=1)
        ax.set_title(name)
        fig.colorbar(im, ax=ax, fraction=0.046)
    fig.suptitle(f'Soil Indices — {scene_folders[0]}')
    fig.tight_layout()

    os.makedirs(reports_dir, exist_ok=True)
    soil_preview_path = os.path.join(reports_dir, f'soil_indices_{scene_folders[0]}.png')
    fig.savefig(soil_preview_path, dpi=120, bbox_inches='tight')
    plt.show(fig)

    try:
        display(Image(filename=soil_preview_path))
    except NameError:
        pass

    print(f'Saved: {soil_preview_path}')
else:
    print('No scene folders found.')

In [ ]:
# ── Fast Parallel Audit ──────────────────────────────────────────────
from concurrent.futures import ThreadPoolExecutor

def validate_directory_fast(directory, scenes, keys, label):
    """Validate expected scene  key files with parallel I/O."""
    if not os.path.isdir(directory):
        print(f'[{label}] directory not found: {directory}')
        return [], [], []

    missing_paths, corrupt_paths = [], []

    def _check_scene(scene):
        miss, corrupt = scan_scene_keys(scene, directory, keys)
        m = [npy_path(directory, scene, k) for k in miss]
        c = [npy_path(directory, scene, k) for k in corrupt]
        return m, c

    # Use parallel threads for I/O bound checks
    with ThreadPoolExecutor(max_workers=12) as executor:
        results = list(tqdm(executor.map(_check_scene, scenes), total=len(scenes), desc=f'Audit {label}'))

    for m, c in results:
        missing_paths.extend(m)
        corrupt_paths.extend(c)

    # Efficient orphan detection using sets
    expected_names = {f'{s}_{k}.npy' for s in scenes for k in keys}
    actual_files = set(os.listdir(directory))
    orphans = [os.path.join(directory, f) for f in actual_files if f.endswith('.npy') and f not in expected_names]

    print(f'[{label}] missing={len(missing_paths)} corrupt={len(corrupt_paths)} orphans={len(orphans)}')
    return missing_paths, corrupt_paths, orphans

def validate_masks_fast(mask_dir, scenes):
    """Parallel audit for masks."""
    if not os.path.isdir(mask_dir):
        return [], []

    def _check_mask(scene):
        path = os.path.join(mask_dir, f'{scene}_mask.npy')
        if not os.path.isfile(path): return path, None
        if not is_valid_npy(path): return None, path
        return None, None

    with ThreadPoolExecutor(max_workers=12) as executor:
        results = list(tqdm(executor.map(_check_mask, scenes), total=len(scenes), desc='Audit masks'))

    missing = [r[0] for r in results if r[0]]
    corrupt = [r[1] for r in results if r[1]]
    print(f'[masks] missing={len(missing)} corrupt={len(corrupt)}')
    return missing, corrupt

# ── Run Optimized Audit ──────────────────────────────────────────────
AUDIT_SCENES = scene_folders if 'scene_folders' in dir() else []
all_missing, all_corrupt, all_orphans = [], [], []

if AUDIT_SCENES:
    # 1. Vegetation
    m, c, o = validate_directory_fast(os.path.join(analysis_dir, 'npy'), AUDIT_SCENES, VEG_KEYS, 'vegetation')
    all_missing += m; all_corrupt += c; all_orphans += o

    # 2. Soil
    m, c, o = validate_directory_fast(os.path.join(analysis_dir, 'soil_npy'), AUDIT_SCENES, SOIL_KEYS, 'soil')
    all_missing += m; all_corrupt += c; all_orphans += o

    # 3. Masks
    mm, mc = validate_masks_fast(masks_dir, AUDIT_SCENES)
    all_missing += mm; all_corrupt += mc

    if not all_missing and not all_corrupt:
        print('\n✅ All cache files are healthy and complete.')
    else:
        print(f'\nAudit Summary: {len(all_missing)} missing, {len(all_corrupt)} corrupt.')
else:
    print('No scenes to validate.')

# Section 5 — Strategi Label & Mask Generation

**Deskripsi:** Karena belum ada ground truth manual, gunakan pseudo-label 4 kelas berbasis threshold NDVI + Soil Index: 0=sehat, 1=rusak ringan, 2=rusak sedang, 3=rusak parah. Validasi visual sample scene dan export mask PNG + NPY.

**Threshold (dapat disesuaikan):**
| Kelas | NDVI | SI |
|-------|------|-----|
| Sehat | > 0.50 | < 0.10 |
| Rusak ringan | 0.35–0.50 | 0.10–0.20 |
| Rusak sedang | 0.20–0.35 | 0.20–0.35 |
| Rusak parah | < 0.20 | > 0.35 |

---


In [ ]:
def generate_damage_mask(ndvi, si, thresholds=THRESHOLDS):
    mask = np.zeros(ndvi.shape, dtype=np.uint8)
    severe = (ndvi < thresholds['ndvi_moderate']) | (
        si > thresholds['si_severe'])
    moderate = ((ndvi >= thresholds['ndvi_moderate']) & (ndvi < thresholds['ndvi_light'])) | (
        (si >= thresholds['si_moderate']) & (si <= thresholds['si_severe'])
    )
    light = ((ndvi >= thresholds['ndvi_light']) & (ndvi < thresholds['ndvi_healthy'])) | (
        (si >= thresholds['si_light']) & (si < thresholds['si_moderate'])
    )
    healthy = (ndvi >= thresholds['ndvi_healthy']) & (
        si < thresholds['si_light'])
    mask[healthy] = 0
    mask[light] = 1
    mask[moderate] = 2
    mask[severe] = 3
    return mask


def mask_to_rgb(mask):
    return CLASS_COLORS[mask]

In [ ]:
os.makedirs(masks_dir, exist_ok=True)
npy_dir = os.path.join(analysis_dir, 'npy')
soil_npy_dir = os.path.join(analysis_dir, 'soil_npy')

# Pre-check: scenes yang sudah selesai di-skip tanpa load apapun
pending = [
    s for s in scene_folders
    if not (
        os.path.isfile(os.path.join(masks_dir, f'{s}_mask.npy')) and
        os.path.isfile(os.path.join(masks_dir, f'{s}_mask.png')) and
        is_valid_npy(os.path.join(masks_dir, f'{s}_mask.npy'))
    )
]
print(f'Skipping {len(scene_folders) - len(pending)} already-done scenes. '
      f'Processing {len(pending)} remaining.')

for scene_name in tqdm(pending, desc='Generating pseudo-labels'):
    mask_npy = os.path.join(masks_dir, f'{scene_name}_mask.npy')
    mask_png = os.path.join(masks_dir, f'{scene_name}_mask.png')
    try:
        # Load satu per satu — bebaskan memori setelah selesai
        ndvi = np.load(os.path.join(npy_dir, f'{scene_name}_NDVI.npy'), mmap_mode='r')
        si   = np.load(os.path.join(soil_npy_dir, f'{scene_name}_SI.npy'), mmap_mode='r')

        mask = generate_damage_mask(ndvi, si)

        # Tutup mmap sebelum lanjut
        del ndvi, si

        np.save(mask_npy, mask)
        PILImage.fromarray(mask_to_rgb(mask)).save(mask_png)

        del mask  # bebaskan segera setelah disimpan

    except Exception as e:
        print(f'Skip mask {scene_name}: {e}')

print(f'Masks saved to: {masks_dir}')

In [ ]:
SAMPLE_N = min(6, len(scene_folders))
sample_scenes = scene_folders[:SAMPLE_N]
npy_dir = os.path.join(analysis_dir, 'npy')
soil_npy_dir = os.path.join(analysis_dir, 'soil_npy')

fig, axes = plt.subplots(SAMPLE_N, 4, figsize=(16, 3 * SAMPLE_N))
if SAMPLE_N == 1:
    axes = np.expand_dims(axes, 0)

for row, scene_name in enumerate(sample_scenes):
    scene_path = os.path.join(dst_folder, scene_name)
    rgb = load_rgb_jpg(scene_path)
    ndvi = np.load(os.path.join(npy_dir, f'{scene_name}_NDVI.npy'))
    si = np.load(os.path.join(soil_npy_dir, f'{scene_name}_SI.npy'))
    mask = np.load(os.path.join(masks_dir, f'{scene_name}_mask.npy'))

    panels = [
        (rgb if rgb is not None else np.stack([ndvi] * 3, axis=-1), 'RGB'),
        (ndvi, 'NDVI'),
        (normalize_index(si), 'SI'),
        (mask_to_rgb(mask) / 255.0, 'Pseudo-label'),
    ]
    for col, (img, title) in enumerate(panels):
        axes[row, col].imshow(
            np.clip(img, 0, 1) if img.ndim == 3 else img,
            cmap=None if img.ndim == 3 else 'RdYlGn',
        )
        axes[row, col].set_title(f'{scene_name}\n{title}', fontsize=9)
        axes[row, col].axis('off')

fig.suptitle('Validasi Visual Pseudo-label (sample scenes)', fontsize=14)
fig.tight_layout()

os.makedirs(reports_dir, exist_ok=True)
preview_path = os.path.join(reports_dir, 'pseudo_label_validation.png')
fig.savefig(preview_path, dpi=120, bbox_inches='tight')
plt.show(fig)

try:
    display(IPyImage(filename=preview_path))
except NameError:
    pass

print(f'Saved: {preview_path}')

# Section 6 — Pembentukan Dataset Segmentasi

**Deskripsi:** Ekstrak patch 256×256 dari NDVI + SI sebagai input multi-channel. Split train/val/test 70/15/15 per scene (hindari data leakage). Augmentasi dengan albumentations. Output folder `dataset/{train,val,test}/{images,masks}`.

---


In [ ]:
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
])

for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(dataset_dir, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(dataset_dir, split, 'masks'), exist_ok=True)


def extract_patches(image, mask, patch_size=PATCH_SIZE, stride=STRIDE):
    h, w = image.shape[:2]
    patches_img, patches_mask = [], []
    for y in range(0, h - patch_size + 1, stride):
        for x in range(0, w - patch_size + 1, stride):
            patches_img.append(image[y:y+patch_size, x:x+patch_size])
            patches_mask.append(mask[y:y+patch_size, x:x+patch_size])
    return patches_img, patches_mask


def build_multichannel(ndvi, si):
    return np.stack([ndvi, normalize_index(si)], axis=-1).astype(np.float32)

In [ ]:
npy_dir = os.path.join(analysis_dir, 'npy')
soil_npy_dir = os.path.join(analysis_dir, 'soil_npy')

train_scenes, temp_scenes = train_test_split(
    scene_folders, test_size=(1 - TRAIN_RATIO), random_state=RANDOM_SEED
)
val_scenes, test_scenes = train_test_split(
    temp_scenes,
    test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
    random_state=RANDOM_SEED,
)
split_map = {'train': train_scenes, 'val': val_scenes, 'test': test_scenes}
print({k: len(v) for k, v in split_map.items()})

patch_counts = {'train': 0, 'val': 0, 'test': 0}

for split_name, scenes in split_map.items():
    img_dir = os.path.join(dataset_dir, split_name, 'images')
    msk_dir = os.path.join(dataset_dir, split_name, 'masks')
    for scene_name in tqdm(scenes, desc=f'Patches {split_name}'):
        try:
            ndvi = np.load(os.path.join(npy_dir, f'{scene_name}_NDVI.npy'))
            si = np.load(os.path.join(soil_npy_dir, f'{scene_name}_SI.npy'))
            mask = np.load(os.path.join(masks_dir, f'{scene_name}_mask.npy'))
            image = build_multichannel(ndvi, si)
            imgs, msks = extract_patches(image, mask)
            for i, (img, msk) in enumerate(zip(imgs, msks)):
                fname = f'{scene_name}_p{i:04d}'
                np.save(os.path.join(img_dir, f'{fname}.npy'), img)
                np.save(os.path.join(
                    msk_dir, f'{fname}.npy'), msk.astype(np.uint8))
                patch_counts[split_name] += 1
        except Exception as e:
            print(f'Skip patch {scene_name}: {e}')

print('Patch counts:', patch_counts)

# Section 7 — Model U-Net (Segmentasi)

**Deskripsi:** Train U-Net dengan backbone ResNet34 (segmentation-models-pytorch). Loss kombinasi CrossEntropy + Dice. Metrik IoU dan Dice per epoch. Simpan checkpoint terbaik ke `models/unet_best.pt`.

---


In [ ]:
print(f'Device: {DEVICE}')

In [ ]:
class PatchDataset(Dataset):
    def __init__(self, split, augment=False, cache=True):
        self.img_dir  = os.path.join(dataset_dir, split, 'images')
        self.msk_dir  = os.path.join(dataset_dir, split, 'masks')
        self.files    = sorted(f for f in os.listdir(self.img_dir) if f.endswith('.npy'))
        self.augment  = augment
        self.transform = train_transform if augment else None

        # Load semua ke RAM sekali — eliminasi disk I/O tiap epoch
        self._cache = None
        if cache:
            self._cache = [
                (np.load(os.path.join(self.img_dir, f)),
                 np.load(os.path.join(self.msk_dir, f)))
                for f in tqdm(self.files, desc=f'Caching {split}', leave=False)
            ]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        if self._cache:
            image, mask = self._cache[idx]
            image, mask = image.copy(), mask.copy()
        else:
            fname = self.files[idx]
            image = np.load(os.path.join(self.img_dir, fname))
            mask  = np.load(os.path.join(self.msk_dir, fname))

        if self.transform:
            out = self.transform(image=image, mask=mask)
            image, mask = out['image'], out['mask']

        image = torch.from_numpy(image.transpose(2, 0, 1)).float()
        mask  = torch.from_numpy(mask).long()
        return image, mask


# ── Metrics: satu kali softmax/argmax untuk dice + iou ───────────────────────

def _per_class_stats(pred_logits, target, num_classes=NUM_CLASSES, eps=1e-7):
    """Return (dice_scores, iou_scores) tensors shape [C]."""
    pred = pred_logits.argmax(dim=1)
    dice, iou = [], []
    for c in range(num_classes):
        p = (pred == c).float()
        t = (target == c).float()
        inter = (p * t).sum()
        card  = p.sum() + t.sum()
        dice.append((2 * inter + eps) / (card + eps))
        iou.append( (inter + eps)     / (card - inter + eps))
    return torch.stack(dice).mean(), torch.stack(iou).mean()



In [ ]:
NUM_WORKERS = min(4, os.cpu_count())
PIN_MEMORY  = DEVICE.type == 'cuda'

# Fungsi fast_cache_load tetap ada namun tidak digunakan untuk loading RAM penuh
def fast_cache_load(img_dir, msk_dir, files, split):
    def load_pair(f):
        return (np.load(os.path.join(img_dir, f)),
                np.load(os.path.join(msk_dir, f)))

    with ThreadPoolExecutor(max_workers=8) as executor:
        results = list(tqdm(executor.map(load_pair, files),
                           total=len(files),
                           desc=f'Caching {split} (Parallel)',
                           leave=False))
    return results

# PatchDataset dikonfigurasi dengan cache=False (baca langsung dari disk)
train_ds = PatchDataset('train', augment=True, cache=False)
val_ds   = PatchDataset('val',   augment=False, cache=False)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0, prefetch_factor=2 if NUM_WORKERS > 0 else None,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0, prefetch_factor=2 if NUM_WORKERS > 0 else None,
)

# ── Model + loss + optimizer ──────────────────────────────────────────────────

unet = smp.Unet(
    encoder_name='resnet34',
    encoder_weights='imagenet',
    in_channels=IN_CHANNELS,
    classes=NUM_CLASSES,
).to(DEVICE)

ce_loss      = nn.CrossEntropyLoss()
dice_loss_fn = smp.losses.DiceLoss(mode='multiclass')
optimizer    = torch.optim.Adam(unet.parameters(), lr=LR)

# Fix: Menggunakan torch.amp.GradScaler untuk kompatibilitas versi terbaru dan CPU/GPU
scaler = torch.amp.GradScaler('cuda', enabled=PIN_MEMORY)

# ── Training loop ─────────────────────────────────────────────────────────────

best_iou     = 0.0
unet_history = {'train_loss': [], 'val_loss': [], 'val_iou': [], 'val_dice': []}

In [ ]:
import torch.backends.cudnn as cudnn
cudnn.benchmark = True  # Optimasi konvolusi PyTorch pada GPU

for epoch in range(EPOCHS):
    # ── Train ──
    unet.train()
    train_loss = 0.0
    pbar = tqdm(train_loader, desc=f'Ep {epoch+1}/{EPOCHS} [Train]', leave=False)
    for images, masks in pbar:
        images, masks = images.to(DEVICE, non_blocking=True), masks.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)           # lebih cepat dari zero_grad()

        with torch.autocast(device_type=DEVICE.type, enabled=PIN_MEMORY):  # AMP
            logits = unet(images)
            loss   = ce_loss(logits, masks) + dice_loss_fn(logits, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    train_loss /= max(len(train_loader), 1)

    # ── Validate ──
    unet.eval()
    val_loss = val_iou = val_dice = 0.0
    pbar = tqdm(val_loader, desc=f'Ep {epoch+1}/{EPOCHS} [Val]  ', leave=False)
    with torch.no_grad():
        for images, masks in pbar:
            images, masks = images.to(DEVICE, non_blocking=True), masks.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, enabled=PIN_MEMORY):
                logits = unet(images)
                loss   = ce_loss(logits, masks) + dice_loss_fn(logits, masks)
            dice, iou   = _per_class_stats(logits.float(), masks)   # satu pass
            val_loss   += loss.item()
            val_iou    += iou.item()
            val_dice   += dice.item()
            pbar.set_postfix({'val_loss': f'{loss.item():.4f}'})

    n = max(len(val_loader), 1)
    val_loss /= n; val_iou /= n; val_dice /= n

    unet_history['train_loss'].append(train_loss)
    unet_history['val_loss'].append(val_loss)
    unet_history['val_iou'].append(val_iou)
    unet_history['val_dice'].append(val_dice)

    print(f'Ep {epoch+1:>3}/{EPOCHS} | loss {train_loss:.4f}/{val_loss:.4f} '
          f'| IoU {val_iou:.4f} | Dice {val_dice:.4f}')

    if val_iou > best_iou:
        best_iou = val_iou
        torch.save(unet.state_dict(), os.path.join(models_dir, 'unet_best.pt'))

print(f'Best U-Net val IoU: {best_iou:.4f}')

# Section 8 — (Dihapus pada V2)

**V2:** DeepLabV3 dihilangkan. Fokus pipeline cascade — U-Net (segmentasi pixel) sebagai sumber fitur probabilitas untuk XGBoost (klasifikasi tile). Lihat Section 9.

---


# Section 9 — Pipeline Cascade: U-Net Prob → XGBoost (Klasifikasi Tile)

**Deskripsi:** Untuk setiap tile, U-Net menghasilkan peta probabilitas 4-kelas. Dari peta itu diekstrak **10 fitur** (mean/std tiap channel probabilitas, entropy ketidakpastian, dominant-class confidence) dan digabung dengan **14 fitur tabular** NDVI/SI lama (statistik NDVI/SI, GLCM contrast/homogeneity, histogram NDVI). Label dari majority-vote pseudo-mask. Dilatih **dua** XGBoost untuk ablation: baseline 14 fitur vs cascade 24 fitur. Disimpan ke `models/xgboost_baseline_14f.json` dan `models/xgboost_cascade_24f.json`.

> Prasyarat: Section 7 (U-Net) harus sudah dijalankan sehingga `models/unet_best.pt` tersedia.

---


In [ ]:
BASE_FEATURE_NAMES = [
    'ndvi_mean', 'ndvi_std', 'si_mean', 'si_std',
    'glcm_contrast', 'glcm_homogeneity',
] + [f'ndvi_hist_{k}' for k in range(8)]

UNET_FEATURE_NAMES = []
for _k in PROB_CLASS_KEYS:
    UNET_FEATURE_NAMES += [f'prob_{_k}_mean', f'prob_{_k}_std']
UNET_FEATURE_NAMES += ['entropy_mean', 'dom_conf_mean']

FEATURE_NAMES = BASE_FEATURE_NAMES + UNET_FEATURE_NAMES
assert len(BASE_FEATURE_NAMES) == N_BASE_FEATURES
assert len(UNET_FEATURE_NAMES) == N_UNET_PROB_FEATURES
assert len(FEATURE_NAMES) == N_CASCADE_FEATURES


def extract_base_tile_features(image, mask):
    """14 fitur tabular NDVI/SI (urutan tetap; selalu di depan vektor fitur)."""
    ndvi = image[:, :, 0]
    si = image[:, :, 1]
    ndvi_u8 = np.clip((ndvi + 1) / 2 * 255, 0, 255).astype(np.uint8)
    glcm = graycomatrix(ndvi_u8, distances=[1], angles=[0],
                        levels=256, symmetric=True, normed=True)
    contrast = float(graycoprops(glcm, 'contrast')[0, 0])
    homogeneity = float(graycoprops(glcm, 'homogeneity')[0, 0])
    label = int(np.bincount(mask.flatten(), minlength=NUM_CLASSES).argmax())
    feats = [
        float(ndvi.mean()), float(ndvi.std()),
        float(si.mean()), float(si.std()),
        contrast, homogeneity,
    ]
    hist, _ = np.histogram(ndvi_u8, bins=8, range=(0, 255), density=True)
    feats.extend(hist.tolist())
    return feats, label


def extract_unet_prob_features(prob_hwc):
    """10 fitur dari peta probabilitas U-Net (H, W, NUM_CLASSES) pasca-softmax."""
    eps = 1e-7
    feats = []
    for c in range(NUM_CLASSES):
        ch = prob_hwc[:, :, c]
        feats.append(float(ch.mean()))
        feats.append(float(ch.std()))
    p = np.clip(prob_hwc, eps, 1.0)
    entropy = -np.sum(p * np.log(p), axis=-1)
    feats.append(float(entropy.mean()))      # ketidakpastian U-Net
    dom_conf = prob_hwc.max(axis=-1)
    feats.append(float(dom_conf.mean()))      # dominant-class confidence
    return feats


@torch.no_grad()
def unet_forward_prob(model, image_2ch):
    """Forward satu tile -> peta probabilitas (H, W, NUM_CLASSES)."""
    x = torch.from_numpy(image_2ch.transpose(2, 0, 1)).float().unsqueeze(0).to(DEVICE)
    return torch.softmax(model(x), dim=1).squeeze(0).permute(1, 2, 0).cpu().numpy()


@torch.no_grad()
def build_xgb_dataset(split, model):
    """Bangun matriks fitur cascade 24-d + label per tile untuk satu split."""
    unet_ckpt = os.path.join(models_dir, 'unet_best.pt')
    if not os.path.exists(unet_ckpt):
        raise RuntimeError(
            'models/unet_best.pt tidak ditemukan — jalankan Section 7 (U-Net) dulu.')
    img_dir = os.path.join(dataset_dir, split, 'images')
    msk_dir = os.path.join(dataset_dir, split, 'masks')
    model.eval()
    X, y = [], []
    files = sorted(f for f in os.listdir(img_dir) if f.endswith('.npy'))
    for fname in tqdm(files, desc=f'XGB feats {split}'):
        image = np.load(os.path.join(img_dir, fname))
        mask = np.load(os.path.join(msk_dir, fname))
        base, label = extract_base_tile_features(image, mask)
        prob = unet_forward_prob(model, image)
        pf = extract_unet_prob_features(prob)
        X.append(base + pf)
        y.append(label)
    return np.array(X, dtype=np.float32), np.array(y)


In [ ]:
def make_xgb():
    return XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='multi:softmax',
        num_class=NUM_CLASSES,
        random_state=RANDOM_SEED,
        tree_method='hist',
        device='cuda' if torch.cuda.is_available() else 'cpu',
    )


In [ ]:
# Load U-Net terbaik sebagai feature extractor
unet.load_state_dict(torch.load(os.path.join(
    models_dir, 'unet_best.pt'), map_location=DEVICE))

X_train, y_train = build_xgb_dataset('train', unet)
X_val,   y_val   = build_xgb_dataset('val', unet)
X_test,  y_test  = build_xgb_dataset('test', unet)

assert X_train.shape[1] == N_CASCADE_FEATURES, X_train.shape
print('Feature vector dim:', X_train.shape[1])
print('Feature names:', FEATURE_NAMES)

# Ablation: baseline = slice 14 fitur pertama (tanpa ekstraksi ulang)
configs = [
    ('baseline_14f', slice(0, N_BASE_FEATURES), 'xgboost_baseline_14f.json'),
    ('cascade_24f',  slice(0, N_CASCADE_FEATURES), 'xgboost_cascade_24f.json'),
]
xgb_models = {}
_abl_rows = []
for name, sl, fpath in configs:
    clf = make_xgb()
    clf.fit(X_train[:, sl], y_train)
    xgb_models[name] = clf
    clf.save_model(os.path.join(models_dir, fpath))
    for split_name, X, y in [('val', X_val, y_val), ('test', X_test, y_test)]:
        pred = clf.predict(X[:, sl])
        acc = accuracy_score(y, pred)
        prec, rec, f1, _ = precision_recall_fscore_support(
            y, pred, average='macro', zero_division=0)
        _abl_rows.append({'Model': name, 'Split': split_name,
                          'Accuracy': acc, 'F1': f1,
                          'Precision': prec, 'Recall': rec})
        print(f'XGBoost {name} {split_name}: Acc={acc:.4f} F1={f1:.4f} '
              f'Prec={prec:.4f} Rec={rec:.4f}')

df_ablation = pd.DataFrame(_abl_rows)
display(df_ablation)
df_ablation.to_csv(os.path.join(reports_dir, 'xgb_ablation.csv'), index=False)

# Delta F1 (cascade - baseline) pada test set
_base_f1 = df_ablation.query("Model=='baseline_14f' and Split=='test'")['F1'].iloc[0]
_casc_f1 = df_ablation.query("Model=='cascade_24f' and Split=='test'")['F1'].iloc[0]
print(f'\nDelta macro-F1 (cascade - baseline, test): {_casc_f1 - _base_f1:+.4f}')

# Feature importance model cascade (lihat kontribusi fitur prob U-Net)
df_fi = pd.DataFrame({
    'feature': FEATURE_NAMES,
    'importance': xgb_models['cascade_24f'].feature_importances_,
}).sort_values('importance', ascending=False)
df_fi.to_csv(os.path.join(reports_dir, 'feature_importance_cascade.csv'), index=False)
display(df_fi.head(15))

# Alias untuk section berikutnya
xgb_baseline = xgb_models['baseline_14f']
xgb_cascade = xgb_models['cascade_24f']
print('XGBoost baseline & cascade tersimpan.')


# Section 10 — Visualisasi Hasil & Persentase Kerusakan

**Deskripsi:** Overlay segmentasi pada citra RGB, heatmap NDVI/SI, hitung persentase area per kelas kerusakan per scene. Generate bar chart statistik kerusakan.

---


In [ ]:
def calculate_damage_percentage(mask, class_names=CLASS_NAMES):
    total = mask.size
    stats = {}
    for i, name in enumerate(class_names):
        pct = 100.0 * np.sum(mask == i) / total
        stats[name] = round(pct, 2)
    stats['Total Rusak (%)'] = round(
        100.0 - stats.get(class_names[0], 0), 2
    )
    return stats


def plot_damage_overlay(rgb, pred_mask, alpha=0.45, title=''):
    overlay = rgb.copy() if rgb is not None else np.zeros((*pred_mask.shape, 3))
    color_mask = mask_to_rgb(pred_mask).astype(np.float32) / 255.0
    if overlay.shape[:2] != pred_mask.shape:
        overlay = cv2.resize(overlay, (pred_mask.shape[1], pred_mask.shape[0]))
    blended = np.clip(overlay * (1 - alpha) + color_mask * alpha, 0, 1)
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(np.clip(overlay, 0, 1))
    plt.title('RGB')
    plt.axis('off')
    plt.subplot(1, 2, 2)
    plt.imshow(blended)
    plt.title(title or 'Segmentation Overlay')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    return blended

In [ ]:
@torch.no_grad()
def predict_scene_mask(model, ndvi, si, patch_size=PATCH_SIZE, stride=STRIDE):
    image = build_multichannel(ndvi, si)
    h, w = image.shape[:2]
    pred_accum = np.zeros((h, w, NUM_CLASSES), dtype=np.float32)
    count = np.zeros((h, w), dtype=np.float32)
    model.eval()
    for y in range(0, h - patch_size + 1, stride):
        for x in range(0, w - patch_size + 1, stride):
            patch = image[y:y+patch_size, x:x+patch_size]
            tensor = torch.from_numpy(patch.transpose(
                2, 0, 1)).unsqueeze(0).float().to(DEVICE)
            logits = model(tensor)
            prob = torch.softmax(logits, dim=1).squeeze(
                0).cpu().numpy().transpose(1, 2, 0)
            pred_accum[y:y+patch_size, x:x+patch_size] += prob
            count[y:y+patch_size, x:x+patch_size] += 1
    count = np.maximum(count, 1)
    return pred_accum / count[..., None]


if scene_folders and os.path.exists(os.path.join(models_dir, 'unet_best.pt')):
    unet.load_state_dict(torch.load(os.path.join(
        models_dir, 'unet_best.pt'), map_location=DEVICE))
    demo = scene_folders[0]
    ndvi = np.load(os.path.join(npy_dir, f'{demo}_NDVI.npy'))
    si = np.load(os.path.join(soil_npy_dir, f'{demo}_SI.npy'))
    rgb = load_rgb_jpg(os.path.join(dst_folder, demo))
    prob = predict_scene_mask(unet, ndvi, si)
    pred = prob.argmax(axis=-1).astype(np.uint8)
    stats = calculate_damage_percentage(pred)
    print(f'Damage stats — {demo}:', stats)
    plot_damage_overlay(rgb, pred, title=f'U-Net Prediction — {demo}')

In [ ]:
# === Visualisasi tile-level cascade: pseudo-label vs U-Net vs XGBoost ===
if (scene_folders and 'xgb_cascade' in dir()
        and os.path.exists(os.path.join(models_dir, 'unet_best.pt'))):
    demo = scene_folders[0]
    ndvi = np.load(os.path.join(npy_dir, f'{demo}_NDVI.npy'))
    si = np.load(os.path.join(soil_npy_dir, f'{demo}_SI.npy'))
    image = build_multichannel(ndvi, si)
    mask_demo = np.load(os.path.join(masks_dir, f'{demo}_mask.npy'))

    prob = predict_scene_mask(unet, ndvi, si)          # HxWx4 (reuse)
    unet_pred = prob.argmax(axis=-1).astype(np.uint8)

    h, w = image.shape[:2]
    xgb_tile = np.zeros((h, w), dtype=np.uint8)
    for yy in range(0, h - PATCH_SIZE + 1, STRIDE):
        for xx in range(0, w - PATCH_SIZE + 1, STRIDE):
            img_win = image[yy:yy+PATCH_SIZE, xx:xx+PATCH_SIZE]
            msk_win = mask_demo[yy:yy+PATCH_SIZE, xx:xx+PATCH_SIZE]
            prob_win = prob[yy:yy+PATCH_SIZE, xx:xx+PATCH_SIZE]
            base, _ = extract_base_tile_features(img_win, msk_win)
            pf = extract_unet_prob_features(prob_win)
            feat = np.array(base + pf, dtype=np.float32).reshape(1, -1)
            xgb_tile[yy:yy+PATCH_SIZE, xx:xx+PATCH_SIZE] = int(
                xgb_cascade.predict(feat)[0])

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    for ax, m, ttl in [
        (axes[0], mask_demo, 'Pseudo-label'),
        (axes[1], unet_pred, 'U-Net argmax'),
        (axes[2], xgb_tile, 'XGBoost cascade (tile)'),
    ]:
        ax.imshow(mask_to_rgb(m))
        ax.set_title(f'{ttl}\n{calculate_damage_percentage(m)}', fontsize=8)
        ax.axis('off')
    plt.suptitle(f'Perbandingan Segmentasi — {demo}', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(reports_dir, 'cascade_tile_comparison.png'),
                dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('Lewati visual cascade: butuh xgb_cascade (Section 9) & unet_best.pt.')


In [ ]:
damage_rows = []
for scene_name in tqdm(scene_folders[:50], desc='Damage stats (first 50 scenes)'):
    mask_path = os.path.join(masks_dir, f'{scene_name}_mask.npy')
    if not os.path.exists(mask_path):
        continue
    mask = np.load(mask_path)
    row = {'Scene': scene_name}
    row.update(calculate_damage_percentage(mask))
    damage_rows.append(row)

if damage_rows:
    df_damage = pd.DataFrame(damage_rows)
    df_damage.to_csv(os.path.join(
        reports_dir, 'damage_percentage_by_scene.csv'), index=False)
    plt.figure(figsize=(12, 5))
    plt.bar(df_damage['Scene'], df_damage['Total Rusak (%)'], color='coral')
    plt.xticks(rotation=90, fontsize=6)
    plt.ylabel('Total Rusak (%)')
    plt.title('Persentase Kerusakan per Scene (pseudo-label, sample 50)')
    plt.tight_layout()
    plt.show()

# Section 11 — Evaluasi & Perbandingan Model

**Deskripsi:** Evaluasi U-Net pada test set (IoU, Dice, Accuracy pixel-level). Bandingkan XGBoost baseline 14f vs cascade 24f (Accuracy, macro-F1 tile-level). Simpan tabel ke `reports/evaluation_summary.csv`.

---


In [ ]:
@torch.no_grad()
def evaluate_segmentation_model(model, loader):
    model.eval()
    ious, dices, accs = [], [], []
    for images, masks in loader:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        logits = model(images)
        dice, iou = _per_class_stats(logits.float(), masks)
        ious.append(iou.item())
        dices.append(dice.item())
        pred = torch.softmax(logits, dim=1).argmax(dim=1)
        accs.append((pred == masks).float().mean().item())
    n = max(len(loader), 1)
    return sum(ious)/n, sum(dices)/n, sum(accs)/n


test_ds = PatchDataset('test', augment=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=0)

results = []

# U-Net (pixel-level)
if os.path.exists(os.path.join(models_dir, 'unet_best.pt')):
    unet.load_state_dict(torch.load(os.path.join(
        models_dir, 'unet_best.pt'), map_location=DEVICE))
    ui, ud, ua = evaluate_segmentation_model(unet, test_loader)
    results.append({'Model': 'U-Net (pixel)', 'EvaluationLevel': 'pixel',
                    'Accuracy': ua, 'IoU': ui, 'Dice': ud, 'F1': ud})

# XGBoost (tile-level): baseline 14f + cascade 24f
if 'X_test' not in dir() or 'y_test' not in dir():
    X_test, y_test = build_xgb_dataset('test', unet)
_eval_specs = [
    ('XGBoost baseline 14f', 'baseline_14f', slice(0, N_BASE_FEATURES)),
    ('XGBoost cascade 24f',  'cascade_24f',  slice(0, N_CASCADE_FEATURES)),
]
for label, key, sl in _eval_specs:
    clf = xgb_models.get(key) if 'xgb_models' in dir() else None
    if clf is None:
        continue
    pred_test = clf.predict(X_test[:, sl])
    acc = sk_acc(y_test, pred_test)
    _, _, f1, _ = precision_recall_fscore_support(
        y_test, pred_test, average='macro', zero_division=0)
    results.append({'Model': label, 'EvaluationLevel': 'patch',
                    'Accuracy': acc, 'IoU': None, 'Dice': None, 'F1': f1})

df_eval = pd.DataFrame(results)
display(df_eval)
df_eval.to_csv(os.path.join(
    reports_dir, 'evaluation_summary.csv'), index=False)
print(f'Evaluation saved to {reports_dir}/evaluation_summary.csv')


## 11.1 — Confusion Matrix per Model

> **Catatan metodologi:** Confusion matrix U-Net dihitung **pixel-level**, sedangkan XGBoost **tile-level** (granularitas berbeda). Perbandingan apple-to-apple yang valid adalah **baseline-14f vs cascade-24f** (sama-sama tile), untuk mengukur kontribusi fitur probabilitas U-Net.

---


In [ ]:
def plot_confusion(cm, class_names, title, ax=None, normalize=True):
    cm_arr = cm.astype(np.float32)
    if normalize:
        cm_arr = cm_arr / (cm_arr.sum(axis=1, keepdims=True) + 1e-9)
    else:
        cm_arr = cm_arr.astype(int)  # ← fix: cast to int for raw counts
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm_arr,
        annot=True,
        fmt='.2f' if normalize else 'd',
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names,
        ax=ax,
        cbar=True,
    )
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(title)
    return ax


@torch.no_grad()
def collect_seg_predictions(model, loader):
    model.eval()
    all_pred, all_true = [], []
    for images, masks in loader:
        images = images.to(DEVICE)
        logits = model(images)
        pred = torch.softmax(logits, dim=1).argmax(dim=1).cpu().numpy()
        all_pred.append(pred.flatten())
        all_true.append(masks.numpy().flatten())
    if not all_pred:
        return np.array([]), np.array([])
    return np.concatenate(all_true), np.concatenate(all_pred)

In [ ]:
import matplotlib.pyplot as plt
import gc
import ctypes
import torch
%matplotlib inline

def _free_ram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    try:
        ctypes.cdll.LoadLibrary('libc.so.6').malloc_trim(0)
    except:
      pass

test_ds = PatchDataset('test', augment=False, cache=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=0)

In [ ]:
_free_ram()

In [ ]:
import gc, torch
import numpy as np
from sklearn.metrics import confusion_matrix
from tqdm.auto import tqdm

# ── Incremental CM helper (pixel-level untuk segmentasi) ──
def incremental_cm(model, loader, num_classes, device, model_name="model"):
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    model.eval()
    pbar = tqdm(loader, desc=f"  {model_name}", unit="batch",
                dynamic_ncols=True, colour="cyan")
    total_px = 0
    with torch.no_grad():
        for batch in pbar:
            imgs = batch[0].to(device, non_blocking=True)
            masks = batch[1].numpy().ravel()
            logits = model(imgs)
            if isinstance(logits, dict):
                logits = logits['out']
            preds = logits.argmax(1).cpu().numpy().ravel()
            valid = (masks >= 0) & (masks < num_classes)
            cm += confusion_matrix(masks[valid], preds[valid],
                                   labels=list(range(num_classes)))
            total_px += valid.sum()
            acc = np.diag(cm).sum() / cm.sum() if cm.sum() > 0 else 0.0
            pbar.set_postfix(pixels=f"{total_px:,}", acc=f"{acc:.3f}")
            del imgs, logits, preds
            torch.cuda.empty_cache()
    pbar.close()
    return cm


def _purge(*var_names, scope):
    for name in var_names:
        if name in scope:
            del scope[name]
    gc.collect()
    torch.cuda.empty_cache()


# ── Evaluation loop ──
_cm_results = {}
_g = globals()

MODELS = [
    ("U-Net", "unet", "unet_best.pt"),
]

outer = tqdm(MODELS, desc="Evaluating models", unit="model",
             dynamic_ncols=True, colour="green")
for label, var, fname in outer:
    outer.set_description(f"Evaluating → {label}")
    path = os.path.join(models_dir, fname)
    if not os.path.exists(path):
        tqdm.write(f"  [skip] {fname} not found")
        continue
    tqdm.write(f"\n[{label}] loading weights …")
    _g[var].load_state_dict(torch.load(path, map_location=DEVICE))
    _cm_results[label] = incremental_cm(_g[var], test_loader,
                                        NUM_CLASSES, DEVICE, label)
    acc = np.diag(_cm_results[label]).sum() / _cm_results[label].sum()
    tqdm.write(f"  ✓ {label} done — overall acc: {acc:.4f}")
    _purge(var, scope=_g)
outer.close()

# ── XGBoost (tile-level): baseline + cascade ──
_xgb_specs = [
    ('XGBoost-14f', 'xgboost_baseline_14f.json', slice(0, N_BASE_FEATURES)),
    ('XGBoost-24f', 'xgboost_cascade_24f.json',  slice(0, N_CASCADE_FEATURES)),
]
if any(os.path.exists(os.path.join(models_dir, f)) for _, f, _ in _xgb_specs):
    if 'X_test' not in _g or 'y_test' not in _g:
        if 'unet' not in _g:
            unet = smp.Unet(encoder_name='resnet34', encoder_weights=None,
                            in_channels=IN_CHANNELS, classes=NUM_CLASSES).to(DEVICE)
            unet.load_state_dict(torch.load(os.path.join(
                models_dir, 'unet_best.pt'), map_location=DEVICE))
        X_test, y_test = build_xgb_dataset('test', unet)
    for _name, _f, _sl in _xgb_specs:
        _p = os.path.join(models_dir, _f)
        if not os.path.exists(_p):
            continue
        tqdm.write(f"\n[{_name}] loading …")
        _xgb_eval = XGBClassifier()
        _xgb_eval.load_model(_p)
        cm_xgb = confusion_matrix(
            y_test, _xgb_eval.predict(X_test[:, _sl]),
            labels=list(range(NUM_CLASSES)))
        _cm_results[_name] = cm_xgb
        acc = np.diag(cm_xgb).sum() / cm_xgb.sum()
        tqdm.write(f"  ✓ {_name} done — overall acc: {acc:.4f}")
    _purge('X_test', 'y_test', '_xgb_eval', scope=_g)

# ── Summary ──
print(f"\n{'─'*40}")
print(f"{'Model':<20} {'Overall Acc':>12}")
print(f"{'─'*40}")
for name, cm in _cm_results.items():
    acc = np.diag(cm).sum() / cm.sum()
    print(f"{name:<20} {acc:>11.4f}")
print(f"{'─'*40}")


In [ ]:
if _cm_results:
    n_models = len(_cm_results)
    fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 5))
    if n_models == 1:
        axes = [axes]
    for ax, (name, cm) in zip(axes, _cm_results.items()):
        plot_confusion(cm, CLASS_NAMES,
                       f'{name} (normalized)', ax=ax, normalize=True)
    plt.suptitle(
        'Confusion Matrix Perbandingan Model (Normalized per kelas)', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(
        reports_dir, 'confusion_matrices_normalized.png'), dpi=120, bbox_inches='tight')
    plt.show()

    fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 5))
    if n_models == 1:
        axes = [axes]
    for ax, (name, cm) in zip(axes, _cm_results.items()):
        plot_confusion(cm, CLASS_NAMES,
                       f'{name} (raw counts)', ax=ax, normalize=False)
    plt.suptitle(
        'Confusion Matrix Perbandingan Model (Raw counts)', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(reports_dir, 'confusion_matrices_raw.png'),
                dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('Tidak ada model tersimpan untuk confusion matrix. Jalankan Section 7-9 terlebih dahulu.')

## 11.2 — Classification Report per Model

**Deskripsi:** Precision, Recall, dan F1-Score per kelas untuk masing-masing model, beserta macro/weighted average. Memudahkan analisis kelas mana yang underperform.


In [ ]:
_report_rows = []

if '_cm_results' in globals():
    for name, cm in _cm_results.items():
        print(f'\n=== {name} ===')
        print(f"{'Class':<15} {'Precision':>10} {'Recall':>10} {'F1-Score':>10} {'Support':>10}")

        support = cm.sum(axis=1)
        pred_sum = cm.sum(axis=0)
        diag = np.diag(cm)
        accuracy = diag.sum() / cm.sum() if cm.sum() > 0 else 0.0

        with np.errstate(divide='ignore', invalid='ignore'):
            precision = np.where(pred_sum > 0, diag / pred_sum, 0.0)
            recall = np.where(support > 0, diag / support, 0.0)
            f1 = np.where((precision + recall) > 0, 2 * precision * recall / (precision + recall), 0.0)

        for i, cls in enumerate(CLASS_NAMES):
            print(f"{cls:<15} {precision[i]:>10.4f} {recall[i]:>10.4f} {f1[i]:>10.4f} {support[i]:>10}")
            _report_rows.append({
                'Model': name,
                'Class': cls,
                'Precision': round(float(precision[i]), 4),
                'Recall': round(float(recall[i]), 4),
                'F1': round(float(f1[i]), 4),
                'Support': int(support[i]),
                'Accuracy': round(float(accuracy), 4)
            })
        print(f"\nOverall Accuracy: {accuracy:.4f}")

if _report_rows:
    df_report = pd.DataFrame(_report_rows)
    df_report.to_csv(os.path.join(
        reports_dir, 'classification_report.csv'), index=False)
    print(
        f'\nClassification report disimpan: {reports_dir}/classification_report.csv')


In [ ]:
if '_report_rows' in dir() and _report_rows:
    df_r = pd.DataFrame(_report_rows)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, metric in zip(axes, ['Precision', 'Recall', 'F1']):
        _pivot = df_r.pivot(index='Class', columns='Model', values=metric)
        _pivot.plot(kind='bar', ax=ax, edgecolor='black')
        ax.set_title(f'{metric} per Kelas')
        ax.set_ylabel(metric)
        ax.set_ylim(0, 1)
        ax.grid(True, alpha=0.3, axis='y')
        ax.legend(loc='lower right')
    plt.suptitle('Perbandingan Precision / Recall / F1 per Kelas')
    plt.tight_layout()
    plt.savefig(os.path.join(reports_dir, 'per_class_metrics.png'),
                dpi=120, bbox_inches='tight')
    plt.show()

## 11.2a — Export Training History U-Net

**Deskripsi:** Simpan log epoch U-Net dan plot val IoU / val loss.

---


In [ ]:
if 'unet_history' in dir() and unet_history.get('val_iou'):
    df_th_unet = pd.DataFrame({
        'Model': 'U-Net',
        'Epoch': range(1, len(unet_history['val_iou']) + 1),
        'train_loss': unet_history['train_loss'],
        'val_loss': unet_history['val_loss'],
        'val_iou': unet_history['val_iou'],
        'val_dice': unet_history['val_dice'],
    })
    df_th_unet.to_csv(os.path.join(reports_dir, 'training_history_unet.csv'), index=False)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    ep = df_th_unet['Epoch']
    axes[0].plot(ep, df_th_unet['val_iou'], 'o-')
    axes[0].set_title('U-Net Val IoU')
    axes[0].grid(True, alpha=0.3)
    axes[1].plot(ep, df_th_unet['val_loss'], 'o-')
    axes[1].set_title('U-Net Val Loss')
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(reports_dir, 'training_curves_unet.png'), dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved {reports_dir}/training_history_unet.csv')
else:
    print('Skip: jalankan Section 7 (U-Net) terlebih dahulu.')


## 11.3 — Uji Statistik Cascade (Baseline 14f vs Cascade 24f)

**Deskripsi:** McNemar, Wilcoxon, dan paired t-test pada tile test yang sama. Export `statistical_tests.csv` dan `xgb_tile_predictions.csv`.

---


In [ ]:
from scipy import stats as sp_stats
import segmentation_models_pytorch as smp
import torch

if 'X_test' not in dir() or 'y_test' not in dir():
    if os.path.exists(os.path.join(models_dir, 'unet_best.pt')):
        # Instantiate the U-Net model before loading its state_dict
        unet = smp.Unet(
            encoder_name='resnet34',
            encoder_weights=None, # Weights will be loaded from the saved file
            in_channels=IN_CHANNELS,
            classes=NUM_CLASSES,
        ).to(DEVICE)
        unet.load_state_dict(torch.load(os.path.join(models_dir, 'unet_best.pt'), map_location=DEVICE))
        X_test, y_test = build_xgb_dataset('test', unet)

_clf_base = xgb_models.get('baseline_14f') if 'xgb_models' in dir() else None
_clf_casc = xgb_models.get('cascade_24f') if 'xgb_models' in dir() else None
_stat_rows = []

if _clf_base is not None and _clf_casc is not None and 'y_test' in dir():
    pred_base = _clf_base.predict(X_test[:, :N_BASE_FEATURES])
    pred_casc = _clf_casc.predict(X_test[:, :N_CASCADE_FEATURES])

    df_tiles = pd.DataFrame({
        'tile_idx': np.arange(len(y_test)),
        'y_true': y_test,
        'pred_baseline': pred_base,
        'pred_cascade': pred_casc,
        'baseline_correct': (pred_base == y_test).astype(int),
        'cascade_correct': (pred_casc == y_test).astype(int),
    })
    df_tiles.to_csv(os.path.join(reports_dir, 'xgb_tile_predictions.csv'), index=False)
    print(f'Saved {reports_dir}/xgb_tile_predictions.csv')

    b_ok = df_tiles['baseline_correct'].values.astype(np.int8)
    c_ok = df_tiles['cascade_correct'].values.astype(np.int8)
    n10 = int(np.sum((b_ok == 1) & (c_ok == 0)))
    n01 = int(np.sum((b_ok == 0) & (c_ok == 1)))
    chi2 = (abs(n10 - n01) - 1) ** 2 / max(n10 + n01, 1)
    p_mcn = 1 - sp_stats.chi2.cdf(chi2, df=1)

    acc_b, acc_c = float(b_ok.mean()), float(c_ok.mean())
    _stat_rows.append({
        'Test': 'McNemar tile correct',
        'Comparison': 'baseline_14f vs cascade_24f',
        'n': len(y_test),
        'statistic': chi2,
        'p_value': p_mcn,
        'effect_size': acc_c - acc_b,
        'notes': f"n10={n10}, n01={n01}, acc_base={acc_b:.4f}, acc_casc={acc_c:.4f}",
    })

    # Per-tile accuracy proxy (1=correct, 0=wrong) — Wilcoxon on paired binary
    w_stat, w_p = sp_stats.wilcoxon(c_ok, b_ok, alternative='greater')
    _stat_rows.append({
        'Test': 'Wilcoxon tile correct',
        'Comparison': 'cascade vs baseline',
        'n': len(y_test),
        'statistic': float(w_stat),
        'p_value': float(w_p),
        'effect_size': acc_c - acc_b,
        'notes': 'one-sided cascade > baseline',
    })

    t_stat, t_p = sp_stats.ttest_rel(c_ok, b_ok, alternative='greater')
    _stat_rows.append({
        'Test': 'Paired t-test tile correct',
        'Comparison': 'cascade vs baseline',
        'n': len(y_test),
        'statistic': float(t_stat),
        'p_value': float(t_p),
        'effect_size': acc_c - acc_b,
        'notes': 'one-sided cascade > baseline',
    })

    df_stats = pd.DataFrame(_stat_rows)
    display(df_stats)
    df_stats.to_csv(os.path.join(reports_dir, 'statistical_tests.csv'), index=False)
    print(f'Saved {reports_dir}/statistical_tests.csv')
else:
    print('Skip cascade statistical tests: train Section 9 terlebih dahulu.')

# Section 12 — Kesimpulan & Artefak Output

**Deskripsi:** Ringkasan pipeline cascade (U-Net → XGBoost), lokasi artefak output, dan rekomendasi langkah lanjutan.

**Artefak:**

- `models/unet_best.pt`, `xgboost_baseline_14f.json`, `xgboost_cascade_24f.json`
- `reports/evaluation_summary.csv` — U-Net pixel + XGBoost tile (+ EvaluationLevel)
- `reports/xgb_ablation.csv`, `feature_importance_cascade.csv`
- `reports/statistical_tests.csv` — McNemar baseline vs cascade
- `reports/xgb_tile_predictions.csv`
- `reports/training_history_unet.csv`, `training_curves_unet.png`
- `reports/damage_percentage_by_scene.csv`

**Novelty V2:** Cascade U-Net probability features → XGBoost; ablation 14f vs 24f dengan uji statistik paired.

---



In [ ]:
import sys

output_filename = os.path.join(reports_dir, 'pipeline_summary.txt')

with open(output_filename, 'w') as f:
    sys.stdout = f  # Redirect stdout to the file

    print('=' * 60)
    print('PIPELINE SELESAI (V2 cascade)')
    print('=' * 60)
    print(f'Scenes processed : {len(scene_folders)}')
    print(f'Patch counts     : {patch_counts if "patch_counts" in dir() else "run Section 6"}')
    print(f'Best U-Net IoU   : {best_iou if "best_iou" in dir() else "N/A"}')
    if 'df_ablation' in dir():
        _t = df_ablation[df_ablation.Split == 'test'].set_index('Model')
        for _m in ['baseline_14f', 'cascade_24f']:
            if _m in _t.index:
                print(f'XGBoost {_m:<12} (test): Acc={_t.loc[_m, "Accuracy"]:.4f} '
                      f'F1={_t.loc[_m, "F1"]:.4f}')
    if os.path.exists(os.path.join(reports_dir, 'statistical_tests.csv')):
        print()
        print('Statistical tests (baseline vs cascade):')
        print(pd.read_csv(os.path.join(reports_dir, 'statistical_tests.csv')).to_string(index=False))
    print()
    print('Artefak model:')
    print('  - models/unet_best.pt')
    print('  - models/xgboost_baseline_14f.json')
    print('  - models/xgboost_cascade_24f.json')
    print('Artefak report:')
    print('  - reports/evaluation_summary.csv')
    print('  - reports/xgb_ablation.csv')
    print('  - reports/statistical_tests.csv')
    print('  - reports/xgb_tile_predictions.csv')
    print()
    print('Rekomendasi lanjutan:')
    print('  1. Kalibrasi threshold Section 5 dengan validasi lapangan')
    print('  2. Tambahkan ground truth manual (QGIS export) jika tersedia')
    print('  3. Bandingkan delta cascade vs baseline pada statistical_tests.csv')
    print('=' * 60)

sys.stdout = sys.__stdout__  # Reset stdout to its default
print(f"Output saved to {output_filename}")

In [ ]:
# Optional: archive and download generated reports/models in Google Colab.
from zipfile import ZIP_DEFLATED, ZipFile

archive_path = Path('/content/reports_models.zip') if IN_COLAB else BASE_DIR / 'reports_models.zip'
with ZipFile(archive_path, 'w', compression=ZIP_DEFLATED) as archive:
    for source_dir in (Path(reports_dir), Path(models_dir)):
        if source_dir.exists():
            for file_path in source_dir.rglob('*'):
                if file_path.is_file():
                    archive.write(file_path, file_path.relative_to(BASE_DIR))
print(f'Archive created: {archive_path}')

if IN_COLAB:
    from google.colab import files
    files.download(archive_path)
else:
    print('Automatic browser download is available only in Google Colab.')


In [ ]:
# Optional: copy generated models and reports to Google Drive.
import shutil

if IN_COLAB:
    drive_root = Path('/content/drive/MyDrive/Colab Notebooks/DATA_BARU')
    shutil.copytree(models_dir, drive_root / 'models_v2', dirs_exist_ok=True)
    shutil.copytree(reports_dir, drive_root / 'reports_v2', dirs_exist_ok=True)
    print('Models and reports synchronized to Google Drive.')
else:
    print('Google Drive synchronization skipped outside Colab.')


In [ ]:
import sys
import os

for d in [analysis_dir, masks_dir, dataset_dir, models_dir, reports_dir]:
    os.makedirs(d, exist_ok=True)

def get_dir_size_gb(dir_path):
    total_size = 0
    if os.path.exists(dir_path):
        for dirpath, _, filenames in os.walk(dir_path):
            for f in filenames:
                fp = os.path.join(dirpath, f)
                if not os.path.islink(fp):
                    total_size += os.path.getsize(fp)
    return total_size / (1024**3)

folders_to_check = {
    'dst_folder': dst_folder,
    'analysis_dir': analysis_dir,
    'masks_dir': masks_dir,
    'dataset_dir': dataset_dir,
    'models_dir': models_dir,
    'reports_dir': reports_dir
}

output_text = f'\nRuntime: {"Colab" if IN_COLAB else "Local"}\n'
output_text += f'BASE_DIR: {BASE_DIR}\n'
output_text += f'dst_folder: {dst_folder}\n'
output_text += '\n--- Folder Sizes (GB) ---\n'

for name, path in folders_to_check.items():
    size_gb = get_dir_size_gb(path)
    output_text += f'{name}: {size_gb:.4f} GB\n'

print(output_text)

txt_path = os.path.join(reports_dir, 'folder_sizes.txt')
with open(txt_path, 'w') as f:
    f.write(output_text)

print(f"Output saved to {txt_path}")
sys.stdout.flush()
